In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))


In [2]:
import importlib

import  src.recommenders.popularity as pipeline

importlib.reload(pipeline)

<module 'src.recommenders.popularity' from 'C:\\Users\\rauni\\Documents\\Learning\\Projects\\ads-intelligence-platform\\src\\recommenders\\popularity.py'>

## Popularity Recommender (V1)

In [3]:
from src.recommenders.popularity import PopularityRecommender
from src.pipelines.build_master_dataset import build_master_dataset

master = build_master_dataset(save=False)

model = PopularityRecommender()

model.fit(master)

model.recommend(10)

,rank,video_id,interaction_count
0,1,6921,9110
1,2,2303,7914
2,3,3310,7359
3,4,1868,6406
4,5,7136,5793
5,6,3587,4319
6,7,4636,3936
7,8,6714,3789
8,9,3273,3771
9,10,7177,3726


## Popularity Evaluations

In [4]:
from src.evaluation.popularity_evaluation import PopularityEvaluation

In [5]:
PopularityEvaluation.top_k_coverage(
    master,
    k=20
)

,Metric,Value
0,Top K Interactions,90979.00
1,Percentage of Total Interactions,7.97
2,Coverage %,0.27


In [6]:
PopularityEvaluation.cumulative_popularity(master)

,rank,video_id,interactions,cumulative_interactions,cumulative_interaction_pct,cumulative_video_pct
0,1,6921,9110,9110,0.80,0.01
1,2,2303,7914,17024,1.49,0.03
2,3,3310,7359,24383,2.14,0.04
3,4,1868,6406,30789,2.70,0.05
4,5,7136,5793,36582,3.21,0.07
...,...,...,...,...,...,...
7533,7534,523,1,1141108,100.00,99.95
7534,7535,4239,1,1141109,100.00,99.96
7535,7536,4253,1,1141110,100.00,99.97
7536,7537,4265,1,1141111,100.00,99.99


In [7]:
popularity = PopularityEvaluation.cumulative_popularity(master)

popularity[
    popularity["cumulative_interaction_pct"] >= 50
].head(1)

,rank,video_id,interactions,cumulative_interactions,cumulative_interaction_pct,cumulative_video_pct
527,528,1122,499,570989,50.04,7.0


In [8]:
popularity[
    popularity["cumulative_interaction_pct"] >= 80
].head(1)

,rank,video_id,interactions,cumulative_interactions,cumulative_interaction_pct,cumulative_video_pct
1827,1828,2898,142,912833,80.0,24.25


In [9]:
model.recommend(20)

,rank,video_id,interaction_count
0,1,6921,9110
1,2,2303,7914
2,3,3310,7359
3,4,1868,6406
4,5,7136,5793
5,6,3587,4319
6,7,4636,3936
7,8,6714,3789
8,9,3273,3771
9,10,7177,3726


# Popularity Recommender Evaluation

The popularity recommender is the **first baseline recommendation algorithm** in this project.

Unlike personalized recommendation models, a popularity-based recommender does not consider user preferences or historical behavior. Instead, it simply recommends the videos that have received the highest number of interactions across the entire platform.

Although simple, this baseline is extremely important because every future recommendation model must demonstrate an improvement over it.

---

# Business Question 1

## How much user engagement comes from the Top 20 most popular videos?

### Observation

| Metric | Value |
|---------|------:|
| Top 20 Videos | 20 |
| Total Interactions from Top 20 | 90,979 |
| Percentage of Total Interactions | **7.97%** |
| Catalog Coverage | **0.27%** |

The twenty most popular videos account for approximately **8% of all user interactions**, even though they represent only **0.27% of the entire video catalog**.

---

### Intuition

Imagine a streaming platform with **7,538 videos**.

If we simply recommend the **Top 20** videos to every user, we are recommending less than **1% of the catalog**.

Yet these videos still generate almost **8% of all platform interactions**.

This demonstrates that a small number of videos naturally attract a significant amount of user attention.

However, the remaining **92% of interactions are distributed across thousands of other videos**, indicating that user engagement is **not concentrated entirely on a handful of viral videos**.

---

### Business Impact

This tells us that a simple popularity recommender provides a **strong and competitive baseline**, especially for:

- New users
- Anonymous visitors
- Homepage recommendations
- Cold-start scenarios

However, recommending only these videos would expose users to only a tiny portion of the available catalog, limiting content discovery and personalization.

---

# Business Question 2

## How concentrated is user engagement across the catalog?

### Observation

From the cumulative popularity analysis:

| Interaction Share | Number of Videos | Catalog Percentage |
|------------------:|-----------------:|-------------------:|
| 50% | **529 Videos** | **7.0%** |
| 80% | **1,829 Videos** | **24.25%** |

---

### Intuition

Suppose the platform contains **7,538 videos**.

Out of these:

- Only **529 videos** generate **half of all user interactions**.
- Around **1,829 videos** generate **80% of all interactions**.
- The remaining **5,709 videos** collectively account for only **20% of platform engagement**.

This phenomenon is known as the **Long Tail Distribution**, one of the most fundamental characteristics of recommendation systems.

```
Popular Videos
██████████████████████████

Long Tail
██████████████████████████████████████████████████████████████
```

A relatively small number of videos become highly popular, while a very large number of videos receive comparatively few interactions.

This pattern is observed across almost every modern recommendation platform including YouTube, Netflix, TikTok, Disney+, Spotify, and Amazon.

---

### Business Impact

The long-tail distribution creates one of the biggest challenges in recommendation systems.

If the recommendation engine only promotes already-popular videos:

- Popular content becomes even more popular.
- New content struggles to receive exposure.
- Niche creators receive very little traffic.
- Users miss opportunities to discover diverse content.

A production recommendation system therefore aims to balance:

- User satisfaction
- Content discovery
- Business objectives
- Creator fairness

rather than simply recommending the most popular items.

---

# Business Question 3

## Why is the popularity recommender still useful?

Despite its simplicity, popularity-based recommendation remains one of the most widely used techniques in production systems.

### Advantages

- Extremely fast to compute.
- No model training required.
- Easy to explain.
- Stable recommendations.
- Excellent baseline for evaluating future recommendation models.
- Effective for new users with no interaction history.

---

### Limitations

Popularity recommendations are **identical for every user**.

For example:

User A enjoys Marvel movies.

User B enjoys Disney Princess movies.

The popularity recommender still returns exactly the same recommendations to both users.

It also recommends videos that a user may have already watched multiple times.

Furthermore, popularity ignores temporal dynamics.

A video that became popular several months ago may continue to rank highly even if users are no longer engaging with it today.

---

# Key Takeaways

This evaluation demonstrates several important characteristics of the KuaiRand dataset.

- The dataset exhibits a clear **Long Tail** distribution.
- User engagement is distributed across a large portion of the catalog rather than being dominated by only a few viral videos.
- A popularity recommender provides a strong baseline but is not sufficient for personalized recommendation.
- Future recommendation models should improve upon popularity by considering:
  - User preferences
  - Historical interactions
  - Temporal trends
  - Content similarity

---

# Transition to the Next Phase

The biggest weakness of the current recommender is that **every user receives exactly the same recommendations**.

Our next improvement is therefore straightforward.

Instead of recommending the globally most popular videos, we will recommend:

> **The most popular videos that the user has not already interacted with.**

This will become our first **personalized baseline recommender**, while still remaining simple, fast, and easy to understand.

It also serves as the natural bridge toward collaborative filtering, where recommendations begin to depend on similarities between users and items rather than global popularity alone.

## Popularity Recommender V2 (Exclude Watched Videos)

In [10]:
model = PopularityRecommender()

model.fit(master)

In [11]:
master["user_id"].sample(1)

476115    19995
Name: user_id, dtype: int64

In [12]:
user = 3419

In [13]:
master.loc[
    master.user_id == user,
    "video_id"
].nunique()

314

In [14]:
model.recommend(10)

,rank,video_id,interaction_count
0,1,6921,9110
1,2,2303,7914
2,3,3310,7359
3,4,1868,6406
4,5,7136,5793
5,6,3587,4319
6,7,4636,3936
7,8,6714,3789
8,9,3273,3771
9,10,7177,3726


In [15]:
model.recommend_unseen(
    user,
    master,
    10
)

,rank,video_id,interaction_count
0,1,6921,9110
1,2,3310,7359
2,3,1868,6406
3,4,7136,5793
4,5,3273,3771
5,6,7177,3726
6,7,4955,3701
7,8,4644,3631
8,9,4506,3535
9,10,1725,3504


# Popularity Recommender V2

## Objective

Improve the popularity recommender by avoiding recommendations for videos that the user has already interacted with.

Instead of recommending the same global Top-N videos to everyone, the recommender now filters out videos already seen by the user before generating recommendations.

---

## Business Rule

```
Recommended Videos

=

Most Popular Videos

−

Already Watched Videos
```

---

## Why is this better?

Imagine a user has already watched the most popular video on the platform.

The original popularity recommender would recommend the same video again.

Popularity Recommender V2 removes previously watched videos, allowing users to discover new content while still leveraging globally popular items.

---

## Advantages

- Simple and efficient.
- Personalized recommendations without training an ML model.
- Prevents duplicate recommendations.
- Improves user experience.
- Strong baseline before collaborative filtering.

---

## Remaining Limitations

Although recommendations are now different for different users, they are **still driven only by global popularity**.

For example:

- A Marvel fan and a Disney Princess fan may still receive very similar recommendations.
- User interests are not considered.
- Content similarity is ignored.
- Trending or recent videos are not prioritized.

---

## Next Step

The next improvement is **Collaborative Filtering**.

Instead of asking:

> "What is popular?"

we will ask:

> **"Users who watched similar videos also watched what?"**

This introduces true personalization based on user behavior rather than global popularity.